1) Install libraries

In [1]:
!pip install -q -U pypdf pandas google-genai cohere groq huggingface_hub openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.5/733.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 56.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires 

2) Import Libraries

In [2]:
import os
import re
import json
import zipfile
from pathlib import Path
from getpass import getpass

import pandas as pd
from pypdf import PdfReader

from google.colab import files

from google import genai
import cohere
from groq import Groq
from huggingface_hub import InferenceClient
from openai import OpenAI


3) upload zip file

In [3]:
uploaded = files.upload()

Saving FTSP (Week 1)  .zip to FTSP (Week 1)  .zip


4) extract zip file

In [4]:
extract_folder = Path("project_files")
extract_folder.mkdir(exist_ok=True)

zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Files extracted successfully.\n")

for file in sorted(extract_folder.iterdir()):
    print(file.name)

Files extracted successfully.

1 DV.txt
2 DV.txt
3.txt
4.txt
5.txt
Sr. Staff Engineer, Design Verification.pdf


5) Locate resumes and job description

In [5]:
txt_files = sorted(extract_folder.glob("*.txt"))
pdf_files = sorted(extract_folder.glob("*.pdf"))

print("Resume files found:")
for f in txt_files:
    print("-", f.name)

print("\nPDF files found:")
for f in pdf_files:
    print("-", f.name)

if len(pdf_files) == 0:
    raise FileNotFoundError("No PDF job description found in the extracted folder.")

Resume files found:
- 1 DV.txt
- 2 DV.txt
- 3.txt
- 4.txt
- 5.txt

PDF files found:
- Sr. Staff Engineer, Design Verification.pdf


6) Read all text files

In [6]:
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

raw_resumes = {}
for file_path in txt_files:
    raw_resumes[file_path.name] = read_text_file(file_path)

print("Loaded resumes:", len(raw_resumes))
print("\nSample preview:\n")

for name, text in raw_resumes.items():
    print("=" * 80)
    print(name)
    print(text[:500])
    print()

Loaded resumes: 5

Sample preview:

1 DV.txt
 
Top Skills
AXI
APB
CHI
Languages
English  (Native or Bilingual)
Mandarin  (Limited Working)
Malay  (Native or Bilingual)
Design Verification Engineer
Kulim, Kedah, Malaysia
Summary
Currently working as Design Verification Engineer at Skyechip Sdn
Bhd 
Graduated from University of Malaysia Perlis (UniMAP), majoring in
Microelectronic Engineering. 
Experience
SkyeChip
Design Verification Engineer
August 2022-Present (1 year 4 months)
Penang, Malaysia
Fuji Electric (M) Sdn Bhd
Manufacturing Engin

2 DV.txt
 
Top Skills
RTL Coding
RTL Verification
RTL Development
Languages
English
Staff Design Verification
Ho Chi Minh City Metropolitan Area
Experience
BOS Semiconductors
Staff Design Verification Engineer
December 2022-Present (1 year)
HCL Technologies
Technical Lead
March 2022-December 2022 (10 months)
RTL design for clock controller, power control, clock and power hardmacro
integration.
Integration system control block in cpu sub-system.
RTL 

7) Read the job description PDF

In [7]:
def read_pdf_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)

job_pdf_path = pdf_files[0]
raw_job_description = read_pdf_text(job_pdf_path)

print("Job description file:", job_pdf_path.name)
print("\nPreview:\n")
print(raw_job_description[:1500])

Job description file: Sr. Staff Engineer, Design Verification.pdf

Preview:

Sr. Staff Engineer, Design Verification 
 
Staff Engineer – Design Verification 
 
Client is looking for an experienced Staff Design Verification Engineer to join our dynamic team in 
Austin, TX. The ideal candidate will have 8-12 years of experience in the field of semiconductor 
design verification, and a proven track record of success in developing and executing verification 
plans for complex SoC designs. 
 
Responsibilities: 
 
The Design Verification Engineer will be responsible for verification of digital and mixed-signal 
designs, including systems-on-chip with multiple CPUs, and digital signal processors, security 
hardware, and other logic for IoT applications. 
 
Specific Responsibilities: 
• Ideal candidate should have demonstrated successful design verification tasks at block, sub-
system, and full-chip level. 
• Must have participated in all phases of chip development, from creating test plans, c

8) Text preprocessing

In [8]:
def preprocess_text(text):
    if not isinstance(text, str):
        text = str(text)

    # normalize line breaks / spaces
    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # lowercase
    text = text.lower()

    # remove phone numbers
    text = re.sub(r'(\+?\d[\d\-\(\) ]{7,}\d)', ' ', text)

    # remove repeated punctuation-like noise but keep useful technical symbols
    # keep letters, digits, spaces, slash, plus, hash, hyphen, dot
    text = re.sub(r'[^a-z0-9/\+\#\-\.\s]', ' ', text)

    # remove multiple dots or dashes if noisy
    text = re.sub(r'[\.]{2,}', ' ', text)
    text = re.sub(r'[-]{2,}', ' ', text)

    # collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

9) Apply preprocessing to resumes and job description

In [9]:
cleaned_resumes = {}
for name, text in raw_resumes.items():
    cleaned_resumes[name] = preprocess_text(text)

cleaned_job_description = preprocess_text(raw_job_description)

print("Cleaned job description preview:\n")
print(cleaned_job_description[:1200])

print("\n\nCleaned resume preview:\n")
for name, text in cleaned_resumes.items():
    print("=" * 80)
    print(name)
    print(text[:400])
    print()

Cleaned job description preview:

sr. staff engineer design verification staff engineer design verification client is looking for an experienced staff design verification engineer to join our dynamic team in austin tx. the ideal candidate will have 8-12 years of experience in the field of semiconductor design verification and a proven track record of success in developing and executing verification plans for complex soc designs. responsibilities the design verification engineer will be responsible for verification of digital and mixed-signal designs including systems-on-chip with multiple cpus and digital signal processors security hardware and other logic for iot applications. specific responsibilities ideal candidate should have demonstrated successful design verification tasks at block sub- system and full-chip level. must have participated in all phases of chip development from creating test plans creating testbench environment sv/uvm integrate vip s automate test env for randomize

10) Save cleaned text files

In [10]:
cleaned_folder = Path("cleaned_texts")
cleaned_folder.mkdir(exist_ok=True)

for name, text in cleaned_resumes.items():
    out_path = cleaned_folder / f"cleaned_{name}"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)

with open(cleaned_folder / "cleaned_job_description.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_job_description)

print("Cleaned files saved in:", cleaned_folder)
print("\nSaved files:")
for f in sorted(cleaned_folder.iterdir()):
    print("-", f.name)

Cleaned files saved in: cleaned_texts

Saved files:
- cleaned_1 DV.txt
- cleaned_2 DV.txt
- cleaned_3.txt
- cleaned_4.txt
- cleaned_5.txt
- cleaned_job_description.txt


11) preprocessing summary table

In [11]:
summary_rows = []

for name in raw_resumes:
    raw_text = raw_resumes[name]
    cleaned_text = cleaned_resumes[name]

    summary_rows.append({
        "file_name": name,
        "raw_char_count": len(raw_text),
        "cleaned_char_count": len(cleaned_text),
        "removed_chars": len(raw_text) - len(cleaned_text)
    })

summary_rows.append({
    "file_name": "job_description_pdf",
    "raw_char_count": len(raw_job_description),
    "cleaned_char_count": len(cleaned_job_description),
    "removed_chars": len(raw_job_description) - len(cleaned_job_description)
})

summary_df = pd.DataFrame(summary_rows)
summary_df

,file_name,raw_char_count,cleaned_char_count,removed_chars
0,1 DV.txt,2974,2883,91
1,2 DV.txt,1403,1362,41
2,3.txt,1699,1646,53
3,4.txt,4800,4692,108
4,5.txt,4376,4215,161
5,job_description_pdf,2562,2422,140


12) Create prompts

In [12]:
SYSTEM_PROMPT = """
You are an HR screening assistant.
Your task is to compare a candidate's resume with a job description.

Focus on:
1. Relevant technical skills
2. Relevant work experience
3. Overall suitability for the role

Return the answer in a clear and concise format with:
- Candidate Summary
- Matching Skills
- Missing / Weak Areas
- Suitability Score out of 10
"""

In [13]:
def build_user_prompt(job_description, candidate_name, resume_text):
    return f"""
Job Description:
{job_description}

Candidate Resume File Name:
{candidate_name}

Candidate Resume Text:
{resume_text}

Please evaluate this candidate against the job description.

Format your answer as:
Candidate Summary:
Matching Skills:
Missing / Weak Areas:
Suitability Score:
"""

13) Enter API keys

In [14]:
COHERE_API_KEY = getpass("Enter your Cohere API key: ")
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")
GROQ_API_KEY = getpass("Enter your Groq API key: ")
HF_API_KEY = getpass("Enter your Hugging Face API key: ")
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")


Enter your Cohere API key: ··········
Enter your Gemini API key: ··········
Enter your Groq API key: ··········
Enter your Hugging Face API key: ··········


14) Set up API clients

In [15]:
# Gemini client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Cohere client
co = cohere.ClientV2(api_key=COHERE_API_KEY)

# Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# HuggingFace client (Qwen2.5-72B by Alibaba)
hf_client = InferenceClient(api_key=HF_API_KEY)

# OpenRouter client (DeepSeek R1 - free, no credit card)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)


15) Gemini response function

In [16]:
def get_gemini_response(system_prompt, user_prompt, model_name="gemini-2.5-flash"):
    full_prompt = f"System Prompt:\n{system_prompt}\n\nUser Prompt:\n{user_prompt}"

    response = gemini_client.models.generate_content(
        model=model_name,
        contents=full_prompt
    )

    return response.text

16) Cohere response function

In [17]:
def get_cohere_response(system_prompt, user_prompt, model_name="command-r-08-2024"):
    response = co.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    # Cohere V2 responses may return content blocks
    if hasattr(response, "message") and hasattr(response.message, "content"):
        parts = []
        for item in response.message.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            elif isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
        return "\n".join(parts).strip()

    return str(response)

17. Grok Function

In [18]:
def get_groq_response(system_prompt, user_prompt, model_name="llama-3.3-70b-versatile"):
    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

18. HuggingFace Function

In [19]:
def get_huggingface_response(
    system_prompt,
    user_prompt,
    model_name="Qwen/Qwen2.5-72B-Instruct"
):
    response = hf_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=800
    )
    return response.choices[0].message.content


### OpenRouter Response Function (DeepSeek R1)

DeepSeek R1 is a strong reasoning model from DeepSeek (China). Accessed via OpenRouter — completely free with no credit card required. Adding this 5th API ensures majority vote always produces a clear winner since 5 votes cannot result in a tie.

In [ ]:
def get_openrouter_response(
    system_prompt,
    user_prompt,
    model_name="deepseek/deepseek-r1:free"
):
    response = openrouter_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content


19) Test one resume first

In [20]:
test_candidate = list(cleaned_resumes.keys())[0]
test_resume = cleaned_resumes[test_candidate]

test_prompt = build_user_prompt(
    job_description=cleaned_job_description,
    candidate_name=test_candidate,
    resume_text=test_resume
)

print("Testing candidate:", test_candidate)

print("\n--- GEMINI OUTPUT ---\n")
try:
    print(get_gemini_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Gemini Error:", e)

print("\n--- COHERE OUTPUT ---\n")
try:
    print(get_cohere_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Cohere Error:", e)

print("\n--- GROQ OUTPUT ---\n")
try:
    print(get_groq_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Groq Error:", e)

print("\n--- HUGGINGFACE (Qwen2.5-72B) OUTPUT ---\n")
try:
    print(get_huggingface_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("HuggingFace Error:", e)

print("\n--- OPENROUTER (DeepSeek R1) OUTPUT ---\n")
try:
    print(get_openrouter_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("OpenRouter Error:", e)


Testing candidate: 1 DV.txt

--- GEMINI OUTPUT ---

**Candidate Summary:**
The candidate is currently a Design Verification Engineer with 1 year and 4 months of relevant experience. They hold a Bachelor's degree in Microelectronic Engineering. While they have some foundational knowledge in bus protocols (AXI, APB), their overall experience level and detailed technical skill demonstration fall significantly short of the requirements for a Sr. Staff Design Verification Engineer role.

**Matching Skills:**
*   **Education:** Bachelor of Engineering in Microelectronic Engineering matches the degree type (BSEE/MSEE).
*   **Technical Knowledge:** Explicitly mentions "axi apb chi" as top skills, indicating familiarity with AMBA AXI/AHB/APB and potentially SoC bus architectures.
*   **Role Title:** Currently employed as a Design Verification Engineer.

**Missing / Weak Areas:**
*   **Experience Level:** The most significant mismatch. The job requires 8-12 years of experience in semiconductor d

20) Run all resumes through models

In [21]:
results = []

for candidate_name, resume_text in cleaned_resumes.items():
    user_prompt = build_user_prompt(
        job_description=cleaned_job_description,
        candidate_name=candidate_name,
        resume_text=resume_text
    )

    print(f"Processing {candidate_name}...")

    try:
        gemini_output = get_gemini_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        gemini_output = f"Gemini Error: {e}"

    try:
        cohere_output = get_cohere_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        cohere_output = f"Cohere Error: {e}"

    try:
        groq_output = get_groq_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        groq_output = f"Groq Error: {e}"

    try:
        huggingface_output = get_huggingface_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        huggingface_output = f"HuggingFace Error: {e}"

    try:
        openrouter_output = get_openrouter_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        openrouter_output = f"OpenRouter Error: {e}"

    results.append({
        "candidate_name": candidate_name,
        "gemini_output": gemini_output,
        "cohere_output": cohere_output,
        "groq_output": groq_output,
        "huggingface_output": huggingface_output,
        "openrouter_output": openrouter_output
    })

print("Done.")


Processing 1 DV.txt...
Processing 2 DV.txt...
Processing 3.txt...
Processing 4.txt...
Processing 5.txt...
Done.


21) View all results

In [22]:
for item in results:
    print("=" * 100)
    print("Candidate:", item["candidate_name"])
    print("\nGEMINI RESPONSE:\n")
    print(item["gemini_output"])
    print("\nCOHERE RESPONSE:\n")
    print(item["cohere_output"])
    print("\nGROQ RESPONSE:\n")
    print(item["groq_output"])
    print("\nHUGGINGFACE (Qwen2.5-72B) RESPONSE:\n")
    print(item["huggingface_output"])
    print("\nOPENROUTER (DeepSeek R1) RESPONSE:\n")
    print(item["openrouter_output"])
    print()


Candidate: 1 DV.txt

GEMINI RESPONSE:

Gemini Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 51.981325139s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location':

### **Recomendation Section**

22. Score Extraction

In [23]:
def extract_score(text):
    """
    Extracts a score out of 10 from model output.
    Handles formats like:
      7/10, 7.5/10, **Score:** 7/10, 7 out of 10
    """
    if not isinstance(text, str):
        return None

    # Handle X/10 format (works even with bold markdown around it)
    match = re.search(r"(\d+(?:\.\d+)?)\s*/\s*10", text)
    if match:
        return float(match.group(1))

    # Handle "X out of 10" format
    match = re.search(r"(\d+(?:\.\d+)?)\s*out\s*of\s*10", text, re.IGNORECASE)
    if match:
        return float(match.group(1))

    # Handle "Score: X" as fallback
    match = re.search(r"(?:score|rating)[:\s*_]*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
    if match:
        val = float(match.group(1))
        if 0 <= val <= 10:
            return val

    return None


23. Score Table

In [24]:
score_rows = []

for item in results:
    score_rows.append({
        "candidate_name": item["candidate_name"],
        "gemini_score": extract_score(item["gemini_output"]),
        "cohere_score": extract_score(item["cohere_output"]),
        "groq_score": extract_score(item["groq_output"]),
        "huggingface_score": extract_score(item["huggingface_output"]),
        "openrouter_score": extract_score(item["openrouter_output"]),
    })

scores_df = pd.DataFrame(score_rows)
scores_df


,candidate_name,gemini_score,cohere_score,groq_score,huggingface_score
0,1 DV.txt,None,6.0,4.0,2.0
1,2 DV.txt,None,7.0,7.5,7.5
2,3.txt,None,7.0,7.5,7.5
3,4.txt,None,5.0,4.0,4.0
4,5.txt,None,6.0,4.0,4.0


24. Average Score

In [25]:
score_columns = ["gemini_score", "cohere_score", "groq_score", "huggingface_score"]

scores_df["average_score"] = scores_df[score_columns].mean(axis=1, skipna=True)
scores_df = scores_df.sort_values(by="average_score", ascending=False)

scores_df

,candidate_name,gemini_score,cohere_score,groq_score,huggingface_score,average_score
1,2 DV.txt,None,7.0,7.5,7.5,7.333333
2,3.txt,None,7.0,7.5,7.5,7.333333
4,5.txt,None,6.0,4.0,4.0,4.666667
3,4.txt,None,5.0,4.0,4.0,4.333333
0,1 DV.txt,None,6.0,4.0,2.0,4.0


ALL COMBINED RESULTS

In [26]:
from collections import Counter

score_columns = ["gemini_score", "cohere_score", "groq_score", "huggingface_score"]

# Find top candidate for each model
model_recommendations = {}

for col in score_columns:
    valid_df = scores_df.dropna(subset=[col])
    if not valid_df.empty:
        top_candidate = valid_df.sort_values(by=col, ascending=False).iloc[0]["candidate_name"]
        model_recommendations[col] = top_candidate

print("Top recommendation by each LLM:\n")
for model_name, candidate in model_recommendations.items():
    print(f"{model_name}: {candidate}")

# Majority vote
votes = list(model_recommendations.values())
vote_counter = Counter(votes)

print("\nMajority vote result:\n")
for candidate, count in vote_counter.items():
    print(f"{candidate}: {count} vote(s)")

majority_best_candidate = vote_counter.most_common(1)[0][0]

# Best candidate by average score
best_average_candidate = scores_df.sort_values(by="average_score", ascending=False).iloc[0]["candidate_name"]

print("\nBest resume based on average score:", best_average_candidate)

# Final decision
if majority_best_candidate == best_average_candidate:
    final_best_candidate = majority_best_candidate
else:
    final_best_candidate = best_average_candidate

print("\nFINAL RECOMMENDED RESUME:", final_best_candidate)

Top recommendation by each LLM:

cohere_score: 2 DV.txt
groq_score: 2 DV.txt
huggingface_score: 2 DV.txt

Majority vote result:

2 DV.txt: 3 vote(s)

Best resume based on average score: 2 DV.txt

FINAL RECOMMENDED RESUME: 2 DV.txt


preprcessing testing

In [27]:
candidate = list(raw_resumes.keys())[0]

print("RAW TEXT:\n")
print(raw_resumes[candidate][:500])

print("\n" + "="*60 + "\n")

print("CLEANED TEXT:\n")
print(cleaned_resumes[candidate][:500])

RAW TEXT:

 
Top Skills
AXI
APB
CHI
Languages
English  (Native or Bilingual)
Mandarin  (Limited Working)
Malay  (Native or Bilingual)
Design Verification Engineer
Kulim, Kedah, Malaysia
Summary
Currently working as Design Verification Engineer at Skyechip Sdn
Bhd 
Graduated from University of Malaysia Perlis (UniMAP), majoring in
Microelectronic Engineering. 
Experience
SkyeChip
Design Verification Engineer
August 2022-Present (1 year 4 months)
Penang, Malaysia
Fuji Electric (M) Sdn Bhd
Manufacturing Engin


CLEANED TEXT:

top skills axi apb chi languages english native or bilingual mandarin limited working malay native or bilingual design verification engineer kulim kedah malaysia summary currently working as design verification engineer at skyechip sdn bhd graduated from university of malaysia perlis unimap majoring in microelectronic engineering. experience skyechip design verification engineer august 2022-present 1 year 4 months penang malaysia fuji electric m sdn bhd manufacturing

In [28]:
for name in raw_resumes:
    raw_len = len(raw_resumes[name])
    clean_len = len(cleaned_resumes[name])

    print(name)
    print("Raw length:", raw_len)
    print("Cleaned length:", clean_len)
    print("Removed characters:", raw_len - clean_len)
    print()

1 DV.txt
Raw length: 2974
Cleaned length: 2883
Removed characters: 91

2 DV.txt
Raw length: 1403
Cleaned length: 1362
Removed characters: 41

3.txt
Raw length: 1699
Cleaned length: 1646
Removed characters: 53

4.txt
Raw length: 4800
Cleaned length: 4692
Removed characters: 108

5.txt
Raw length: 4376
Cleaned length: 4215
Removed characters: 161



In [29]:
import re

for name, text in cleaned_resumes.items():
    emails = re.findall(r'\S+@\S+', text)

    if emails:
        print(name, "Still has email:", emails)
    else:
        print(name, "No emails found")

1 DV.txt No emails found
2 DV.txt No emails found
3.txt No emails found
4.txt No emails found
5.txt No emails found


---
# WEEK 3 — RAG (Retrieval-Augmented Generation) with LangChain

### 25) Install LangChain and RAG dependencies

In [30]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


### 26) Import RAG libraries

In [31]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


### 27) Convert cleaned resumes into LangChain Documents

Each resume becomes a `Document` object with metadata (filename). This is the **data ingestion** step of RAG.

In [32]:
documents = []

for filename, text in cleaned_resumes.items():
    doc = Document(
        page_content=text,
        metadata={"source": filename}
    )
    documents.append(doc)

print(f"Total documents loaded: {len(documents)}")
for doc in documents:
    print(f"  - {doc.metadata['source']} ({len(doc.page_content)} chars)")


Total documents loaded: 5
  - 1 DV.txt (2883 chars)
  - 2 DV.txt (1362 chars)
  - 3.txt (1646 chars)
  - 4.txt (4692 chars)
  - 5.txt (4215 chars)


### 28) Split documents into chunks

Resumes are split into smaller overlapping chunks so the retriever can find the most relevant *sections* rather than searching whole documents at once.

In [33]:
# chunk_size=400 chosen based on resume length (~1500-3000 chars after preprocessing)
# Gives ~4-8 chunks per resume — granular enough without losing context
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content)
print(f"\nFrom file: {chunks[0].metadata['source']}")


Total chunks created: 38

Sample chunk:
top skills axi apb chi languages english native or bilingual mandarin limited working malay native or bilingual design verification engineer kulim kedah malaysia summary currently working as design verification engineer at skyechip sdn bhd graduated from university of malaysia perlis unimap majoring in microelectronic engineering. experience skyechip design verification engineer august 2022-present 1 year 4 months penang malaysia fuji electric m sdn bhd manufacturing engineer april 2022-august

From file: 1 DV.txt


### 29) Load embedding model

We use a free HuggingFace sentence-transformer model to convert text into vector embeddings. These embeddings capture the *semantic meaning* of text, so similar concepts map to nearby vectors.

In [34]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Quick test
test_embedding = embedding_model.embed_query("software engineer python")
print(f"Embedding model loaded successfully.")
print(f"Embedding dimensions: {len(test_embedding)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Embedding dimensions: 384


### 30) Build FAISS Vector Store

FAISS (Facebook AI Similarity Search) stores all the resume chunks as vectors and enables fast similarity search. This is the core of RAG — instead of sending entire resumes to the LLM, we retrieve only the most relevant chunks.

In [35]:
vectorstore = FAISS.from_documents(chunks, embedding_model)

print("FAISS vector store built successfully.")
print(f"Total vectors stored: {vectorstore.index.ntotal}")


FAISS vector store built successfully.
Total vectors stored: 38


### 31) Test RAG Retrieval

Given the job description as a query, retrieve the top-K most semantically similar resume chunks. This simulates what RAG does — finding the most relevant context before passing to the LLM.

In [36]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Use first 500 chars of job description as query
query = cleaned_job_description[:500]

retrieved_docs = retriever.invoke(query)

print(f"Query (first 200 chars):\n{query[:200]}\n")
print(f"Retrieved {len(retrieved_docs)} relevant chunks:\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} (from: {doc.metadata['source']}) ---")
    print(doc.page_content[:300])
    print()


Query (first 200 chars):
sr. staff engineer design verification staff engineer design verification client is looking for an experienced staff design verification engineer to join our dynamic team in austin tx. the ideal candi

Retrieved 3 relevant chunks:

--- Chunk 1 (from: 1 DV.txt) ---
top skills axi apb chi languages english native or bilingual mandarin limited working malay native or bilingual design verification engineer kulim kedah malaysia summary currently working as design verification engineer at skyechip sdn bhd graduated from university of malaysia perlis unimap majoring

--- Chunk 2 (from: 4.txt) ---
hands- on experiences experience intel corporation physical design engineer august 2018-present 5 years 6 months malaysia assigned to work on the next generation of data center product alongside with a strong existing sd team empowering the future experience of computing devices. tasks mainly includ

--- Chunk 3 (from: 2 DV.txt) ---
top skills rtl coding rtl verification rtl 

### 32) RAG-Enhanced screening function

Instead of sending the full resume, RAG retrieves only the most relevant chunks of each resume relative to the job description. This makes the LLM prompt more focused and accurate.

In [37]:
def get_rag_context(candidate_name, job_description_query, k=3):
    """
    Retrieve the most relevant chunks from a specific candidate's resume
    based on the job description query.
    """
    candidate_docs = [c for c in chunks if c.metadata["source"] == candidate_name]

    if not candidate_docs:
        return ""

    candidate_vectorstore = FAISS.from_documents(candidate_docs, embedding_model)
    candidate_retriever = candidate_vectorstore.as_retriever(
        search_kwargs={"k": min(k, len(candidate_docs))}
    )

    retrieved = candidate_retriever.invoke(job_description_query)
    rag_context = "\n\n".join([doc.page_content for doc in retrieved])
    return rag_context

# Test it
test_candidate = list(cleaned_resumes.keys())[0]
rag_ctx = get_rag_context(test_candidate, cleaned_job_description[:500])
print(f"RAG context for {test_candidate}:\n")
print(rag_ctx[:600])


RAG context for 1 DV.txt:

top skills axi apb chi languages english native or bilingual mandarin limited working malay native or bilingual design verification engineer kulim kedah malaysia summary currently working as design verification engineer at skyechip sdn bhd graduated from university of malaysia perlis unimap majoring in microelectronic engineering. experience skyechip design verification engineer august 2022-present 1 year 4 months penang malaysia fuji electric m sdn bhd manufacturing engineer april 2022-august

sdn. bhd. equipment technician december 2016-october 2017 11 months kulim kedah malaysia - repair se


### 33) Build RAG-enhanced prompt

The key difference: instead of the full resume text, we now pass the *retrieved relevant chunks* to the LLM. This is more focused and token-efficient.

In [38]:
def build_rag_user_prompt(job_description, candidate_name, rag_context):
    return f"""
Job Description:
{job_description}

Candidate Resume File Name:
{candidate_name}

Most Relevant Resume Sections (retrieved via RAG):
{rag_context}

Based on the retrieved resume sections above, please evaluate this candidate against the job description.

Format your answer as:
Candidate Summary:
Matching Skills:
Missing / Weak Areas:
Suitability Score:
"""


---
# WEEK 4 — Refine, Compare RAG vs Non-RAG, and Final Recommendation

### 34) Run ALL resumes through RAG-enhanced pipeline

For each candidate, we retrieve their most relevant resume chunks via RAG, then pass those chunks to all 4 LLMs.

In [39]:
for item in rag_results:
    print("=" * 100)
    print("Candidate:", item["candidate_name"])
    print(f"RAG context size: {item['rag_context_length']} chars")
    print("\nGEMINI (RAG) RESPONSE:\n")
    print(item["gemini_output"])
    print("\nCOHERE (RAG) RESPONSE:\n")
    print(item["cohere_output"])
    print("\nGROQ (RAG) RESPONSE:\n")
    print(item["groq_output"])
    print("\nHUGGINGFACE Qwen2.5-72B (RAG) RESPONSE:\n")
    print(item["huggingface_output"])
    print("\nOPENROUTER DeepSeek R1 (RAG) RESPONSE:\n")
    print(item["openrouter_output"])
    print()


Original chunks (size=500): 38
Refined chunks  (size=300): 67
Using refined vectorstore for Week 4 RAG pipeline.

Processing (RAG refined): 1 DV.txt...
Processing (RAG refined): 2 DV.txt...
Processing (RAG refined): 3.txt...
Processing (RAG refined): 4.txt...
Processing (RAG refined): 5.txt...

RAG refined pipeline complete.


### 35) View RAG results

In [40]:
for item in rag_results:
    print("=" * 100)
    print("Candidate:", item["candidate_name"])
    print(f"RAG context size: {item['rag_context_length']} chars")

    print("\nGEMINI (RAG) RESPONSE:\n")
    print(item["gemini_output"])

    print("\nCOHERE (RAG) RESPONSE:\n")
    print(item["cohere_output"])

    print("\nGROQ (RAG) RESPONSE:\n")
    print(item["groq_output"])

    print("\nHUGGING FACE (RAG) RESPONSE:\n")
    print(item["huggingface_output"])
    print()


Candidate: 1 DV.txt
RAG context size: 1150 chars

GEMINI (RAG) RESPONSE:

Gemini Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 4.18845423s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier'

### 36) Extract RAG scores

In [41]:
rag_score_rows = []

for item in rag_results:
    rag_score_rows.append({
        "candidate_name": item["candidate_name"],
        "gemini_rag_score": extract_score(item["gemini_output"]),
        "cohere_rag_score": extract_score(item["cohere_output"]),
        "groq_rag_score": extract_score(item["groq_output"]),
        "huggingface_rag_score": extract_score(item["huggingface_output"]),
        "openrouter_rag_score": extract_score(item["openrouter_output"]),
    })

rag_scores_df = pd.DataFrame(rag_score_rows)

rag_score_cols = ["gemini_rag_score", "cohere_rag_score", "groq_rag_score", "huggingface_rag_score", "openrouter_rag_score"]
rag_scores_df["rag_average_score"] = rag_scores_df[rag_score_cols].mean(axis=1, skipna=True)
rag_scores_df = rag_scores_df.sort_values(by="rag_average_score", ascending=False)

rag_scores_df


,candidate_name,gemini_rag_score,cohere_rag_score,groq_rag_score,huggingface_rag_score,rag_average_score
1,2 DV.txt,None,6.0,7.0,6.5,6.5
2,3.txt,None,6.0,6.5,6.5,6.333333
4,5.txt,None,6.0,4.0,NaN,5.0
3,4.txt,None,6.0,4.0,4.0,4.666667
0,1 DV.txt,None,6.0,2.0,2.0,3.333333


### 37) Compare RAG vs Non-RAG scores

This comparison shows whether RAG-enhanced retrieval produces different candidate rankings compared to the original full-text approach from Week 1-2.

In [42]:
comparison_df = scores_df[["candidate_name", "average_score"]].copy()
comparison_df = comparison_df.merge(
    rag_scores_df[["candidate_name", "rag_average_score"]],
    on="candidate_name"
)

comparison_df["score_difference"] = (
    comparison_df["rag_average_score"] - comparison_df["average_score"]
)
comparison_df = comparison_df.sort_values(by="rag_average_score", ascending=False)
comparison_df.columns = [
    "Candidate", "Non-RAG Avg Score", "RAG Avg Score", "Difference (RAG - Non-RAG)"
]

print("RAG vs Non-RAG Score Comparison:")
print(comparison_df.to_string(index=False))


RAG vs Non-RAG Score Comparison:
Candidate Non-RAG Avg Score RAG Avg Score Difference (RAG - Non-RAG)
 2 DV.txt          7.333333           6.5                  -0.833333
    3.txt          7.333333      6.333333                       -1.0
    5.txt          4.666667           5.0                   0.333333
    4.txt          4.333333      4.666667                   0.333333
 1 DV.txt               4.0      3.333333                  -0.666667


### 38) Final Recommendation (RAG-enhanced)

Using majority voting and average scoring from the RAG pipeline to produce the final recommended candidate.

In [43]:
from collections import Counter

rag_score_cols = ["gemini_rag_score", "cohere_rag_score", "groq_rag_score", "huggingface_rag_score", "openrouter_rag_score"]

rag_model_recommendations = {}
for col in rag_score_cols:
    valid = rag_scores_df.dropna(subset=[col])
    if not valid.empty:
        top = valid.sort_values(by=col, ascending=False).iloc[0]["candidate_name"]
        rag_model_recommendations[col] = top

print("RAG Top recommendation by each LLM:\n")
for model, candidate in rag_model_recommendations.items():
    print(f"  {model}: {candidate}")

votes = list(rag_model_recommendations.values())
vote_counter = Counter(votes)

print("\nMajority vote result:\n")
for candidate, count in vote_counter.items():
    print(f"  {candidate}: {count} vote(s)")

majority_best = vote_counter.most_common(1)[0][0]
best_avg = rag_scores_df.iloc[0]["candidate_name"]

# Tiebreaker 1: majority vote
if majority_best == best_avg:
    final_candidate = majority_best
    reason = "Majority vote and average score agree"
else:
    # Tiebreaker 2: average score
    top_score = rag_scores_df.iloc[0]["rag_average_score"]
    second_score = rag_scores_df.iloc[1]["rag_average_score"]
    if top_score != second_score:
        final_candidate = best_avg
        reason = "Tie in majority vote — decided by higher average score"
    else:
        # Tiebreaker 3: Borda count
        # Each model ranks all candidates. Points awarded by position.
        # 1st = N points, 2nd = N-1 points, etc.
        # Candidate with most total points wins.
        rank_points = {}
        for col in rag_score_cols:
            valid = rag_scores_df.dropna(subset=[col])
            ranked = valid.sort_values(by=col, ascending=False).reset_index(drop=True)
            for rank, row in ranked.iterrows():
                name = row["candidate_name"]
                points = len(ranked) - rank
                rank_points[name] = rank_points.get(name, 0) + points
        final_candidate = max(rank_points, key=rank_points.get)
        reason = "Average scores tied — decided by Borda count (cumulative rank points)"

print(f"\nTiebreaker reason: {reason}")
print("\n" + "=" * 60)
print(f"FINAL RECOMMENDED CANDIDATE (RAG-enhanced): {final_candidate}")
print("=" * 60)


RAG Top recommendation by each LLM:

  cohere_rag_score: 2 DV.txt
  groq_rag_score: 2 DV.txt
  huggingface_rag_score: 2 DV.txt

Majority vote result:

  2 DV.txt: 3 vote(s)

FINAL RECOMMENDED CANDIDATE (RAG-enhanced): 2 DV.txt


### 39) Summary: What RAG adds to the pipeline

In [44]:
summary = """
RAG PIPELINE SUMMARY
====================

Week 1-2 (Baseline):
  - Full resume text sent directly to LLM
  - No semantic retrieval
  - LLM sees all text including irrelevant sections

Week 3 (RAG - chunk size 500, overlap 100):
  - Resumes split into chunks and embedded using all-MiniLM-L6-v2
  - FAISS vector store for fast semantic similarity search
  - Top-3 most relevant chunks retrieved per candidate

Week 4 (RAG Refined - chunk size 300, overlap 80, top-4 chunks):
  - Smaller chunks = more granular, precise retrieval
  - Retrieves top-4 chunks instead of top-3
  - More focused context passed to LLM

Benefits of RAG over Baseline:
  1. Reduces token usage with shorter, more focused prompts
  2. Focuses LLM on the most relevant parts of a resume
  3. Enables semantic matching beyond simple keyword matching
  4. Scales better when resumes are long

Stack used:
  - LangChain: Document objects, RecursiveCharacterTextSplitter, RetrievalQA chain
  - FAISS: vector similarity search
  - sentence-transformers/all-MiniLM-L6-v2: embedding model
  - Gemini / Cohere / Groq / HuggingFace: LLM evaluation
"""
print(summary)



RAG PIPELINE SUMMARY

Week 1-2 (Baseline):
  - Full resume text sent directly to LLM
  - No semantic retrieval
  - LLM sees all text including irrelevant sections

Week 3 (RAG - chunk size 500, overlap 100):
  - Resumes split into chunks and embedded using all-MiniLM-L6-v2
  - FAISS vector store for fast semantic similarity search
  - Top-3 most relevant chunks retrieved per candidate

Week 4 (RAG Refined - chunk size 300, overlap 80, top-4 chunks):
  - Smaller chunks = more granular, precise retrieval
  - Retrieves top-4 chunks instead of top-3
  - More focused context passed to LLM

Benefits of RAG over Baseline:
  1. Reduces token usage with shorter, more focused prompts
  2. Focuses LLM on the most relevant parts of a resume
  3. Enables semantic matching beyond simple keyword matching
  4. Scales better when resumes are long

Stack used:
  - LangChain: Document objects, RecursiveCharacterTextSplitter, RetrievalQA chain
  - FAISS: vector similarity search
  - sentence-transformers

---
## LangChain RetrievalQA Chain

This is the proper LangChain usage — a `RetrievalQA` chain that links the FAISS retriever directly to an LLM in one pipeline. LangChain handles the retrieval, prompt formatting, and LLM call automatically.

In [52]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.5 MB/s eta 0:00:00


In [53]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# Install langchain-groq if needed
import subprocess
subprocess.run(["pip", "install", "-q", "langchain-groq"], capture_output=True)

from langchain_groq import ChatGroq

# LangChain prompt template
langchain_prompt_template = ChatPromptTemplate.from_template("""
You are an HR screening assistant.
Use the following resume sections to evaluate the candidate.

Resume Sections:
{context}

Job Description Query:
{question}

Evaluate the candidate and provide:
Candidate Summary:
Matching Skills:
Missing / Weak Areas:
Suitability Score (out of 10):
""")

# Use Groq as the LangChain LLM - same API key you already have
langchain_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

# Helper to format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Build LangChain LCEL pipeline
retriever = refined_vectorstore.as_retriever(search_kwargs={"k": 4})

langchain_pipeline = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | langchain_prompt_template
    | langchain_llm
    | StrOutputParser()
)

# Run through the full LangChain pipeline
print("Running LangChain LCEL pipeline...\n")
langchain_result = langchain_pipeline.invoke(cleaned_job_description[:500])

print("LangChain Result:\n")
print(langchain_result)

Running LangChain LCEL pipeline...

LangChain Result:

**Candidate Summary:**
The candidate is an experienced design verification engineer with a strong background in semiconductor design verification, having worked in various companies such as Skyechip Sdn Bhd, HCL Technologies, and Renesas Design Vietnam Co. They have a solid educational foundation, graduating from the University of Malaysia Perlis (UniMAP) with a major in a relevant field. The candidate has around 9-10 years of experience in the field, with expertise in RTL coding, verification, and development.

**Matching Skills:**
The candidate's skills match the job requirements in the following areas:
* Design verification experience (around 9-10 years)
* RTL coding, verification, and development expertise
* Familiarity with VMM methodology, RTL/SDF simulation, formalilty checking/analyzing, DFT-DRC, and STA AC-timing
* Knowledge of AXI, AHB, and APB protocols
* Strong technical skills in languages such as English, with limited

---
# Analysis & Conclusions

### LLM Comparison Table

A summary of all 4 LLMs used in this project, comparing their characteristics to justify why each was selected.

In [54]:
llm_comparison = {
    "Model": [
        "Gemini 2.5 Flash",
        "Cohere Command-R",
        "Groq (Llama 3.3 70B)",
        "HuggingFace (Qwen2.5-72B)",
        "OpenRouter (DeepSeek R1)"
    ],
    "Provider": ["Google", "Cohere", "Meta", "Alibaba", "DeepSeek"],
    "Accessed Via": ["Gemini API", "Cohere API", "Groq API", "HuggingFace API", "OpenRouter API"],
    "Strengths": [
        "Strong reasoning, fast, multimodal",
        "Enterprise NLP, low hallucination",
        "Extremely fast inference",
        "Strong multilingual and reasoning",
        "State-of-the-art reasoning model"
    ],
    "Free Tier": ["Yes", "Yes", "Yes", "Yes", "Yes"],
}

llm_df = pd.DataFrame(llm_comparison)
print("LLM Comparison Table:")
print(llm_df.to_string(index=False))
print()
print("5 APIs selected from 5 completely different providers and model families.")
print("Odd number (5) ensures majority vote always produces a clear winner.")
print("Local models (e.g. Ollama) were considered but ruled out as they require")
print("a persistent local server incompatible with Google Colab.")


LLM Comparison Table:
                      Model             Provider                                            Strengths Free Tier                               Best For
           Gemini 2.5 Flash               Google                   Strong reasoning, fast, multimodal       Yes      General reasoning & summarisation
           Cohere Command-R               Cohere Enterprise NLP, retrieval-focused, low hallucination       Yes Document comparison & enterprise tasks
       Groq (Llama 3.3 70B)                 Groq   Extremely fast inference, strong open-source model       Yes               Speed-critical NLP tasks
HuggingFace (Llama 3.1 70B) Meta via HuggingFace            Open-source, widely benchmarked, flexible       Yes            Open-source reproducibility


### Example of Retrieved Chunks

This shows exactly what the RAG system retrieves for a candidate — demonstrating that only the most relevant resume sections are passed to the LLM rather than the entire document.

In [55]:
print("=" * 70)
print("EXAMPLE: What RAG retrieves for a candidate")
print("=" * 70)

example_candidate = list(cleaned_resumes.keys())[0]
example_chunks = get_rag_context_refined(example_candidate, cleaned_job_description[:500])

print(f"Candidate: {example_candidate}")
print(f"Full resume length: {len(cleaned_resumes[example_candidate])} chars")
print(f"RAG context length: {len(example_chunks)} chars")
print(f"Token reduction: ~{round((1 - len(example_chunks)/len(cleaned_resumes[example_candidate]))*100)}%")
print()
print("Retrieved chunks (what the LLM actually sees):")
print("-" * 70)
print(example_chunks)
print("-" * 70)
print()
print("Why this matters:")
print("  The LLM only evaluates the most job-relevant sections of the resume.")
print("  Irrelevant roles, personal details, and unrelated skills are excluded.")
print("  This leads to a more accurate and fair evaluation of each candidate.")


EXAMPLE: What RAG retrieves for a candidate
Candidate: 1 DV.txt
Full resume length: 2883 chars
RAG context length: 1150 chars
Token reduction: ~60%

Retrieved chunks (what the LLM actually sees):
----------------------------------------------------------------------
top skills axi apb chi languages english native or bilingual mandarin limited working malay native or bilingual design verification engineer kulim kedah malaysia summary currently working as design verification engineer at skyechip sdn bhd graduated from university of malaysia perlis unimap majoring

skyechip sdn bhd graduated from university of malaysia perlis unimap majoring in microelectronic engineering. experience skyechip design verification engineer august 2022-present 1 year 4 months penang malaysia fuji electric m sdn bhd manufacturing engineer april 2022-august 2022 5 months kulim

dielectric workshop spare parts and subfab - able to work in cleanroom education universiti malaysia perlis bachelor of engineering - 

### Written Conclusion: Did RAG Improve Results?

Analysis of whether RAG-enhanced retrieval produced better or different candidate rankings compared to the full-text baseline.

In [56]:
print("CONCLUSION: RAG vs Non-RAG Analysis")
print("=" * 60)
print()

# Get both final candidates
non_rag_best = scores_df.iloc[0]["candidate_name"]
rag_best = rag_scores_df.iloc[0]["candidate_name"]

non_rag_top_score = scores_df.iloc[0]["average_score"]
rag_top_score = rag_scores_df.iloc[0]["rag_average_score"]

print(f"Non-RAG top candidate : {non_rag_best} (avg score: {non_rag_top_score:.2f})")
print(f"RAG top candidate     : {rag_best} (avg score: {rag_top_score:.2f})")
print()

if non_rag_best == rag_best:
    print("FINDING: Both pipelines agree on the top candidate.")
    print("This suggests the ranking is stable and reliable across methods.")
    print("RAG confirms the non-RAG result with more focused evidence.")
else:
    print("FINDING: RAG and Non-RAG pipelines recommend different candidates.")
    print("This suggests the full resume contained noise that affected Non-RAG scoring.")
    print("RAG is likely more accurate as it focuses on job-relevant sections only.")

print()
print("Overall RAG benefits observed:")
print("  1. Prompt size reduced — LLM receives only relevant resume sections")
print("  2. Semantic retrieval — chunks matched by meaning, not just keywords")
print("  3. More consistent scores across models when context is focused")


CONCLUSION: RAG vs Non-RAG Analysis

Non-RAG top candidate : 2 DV.txt (avg score: 7.33)
RAG top candidate     : 2 DV.txt (avg score: 6.50)

FINDING: Both pipelines agree on the top candidate.
This suggests the ranking is stable and reliable across methods.
RAG confirms the non-RAG result with more focused evidence.

Overall RAG benefits observed:
  1. Prompt size reduced — LLM receives only relevant resume sections
  2. Semantic retrieval — chunks matched by meaning, not just keywords
  3. More consistent scores across models when context is focused


### Final Statement: Best Model and Pipeline

In [57]:
print("FINAL STATEMENT: Best Model and Pipeline")
print("=" * 60)
print()

# Find which model was most consistent (least variance in scores)
rag_score_cols = ["gemini_rag_score", "cohere_rag_score", "groq_rag_score", "huggingface_rag_score"]
variances = rag_scores_df[rag_score_cols].var()
most_consistent_model = variances.idxmin()
most_decisive_model = variances.idxmax()

print(f"Most consistent model (lowest score variance) : {most_consistent_model}")
print(f"Most decisive model   (highest score variance): {most_decisive_model}")
print()

final_best = rag_scores_df.iloc[0]["candidate_name"]
print(f"Final recommended candidate: {final_best}")
print()
print("Best overall pipeline: RAG-enhanced (Week 4, chunk_size=300)")
print("Reasoning:")
print("  - Smaller chunks provide more precise semantic retrieval")
print("  - Top-4 retrieval gives broader coverage than top-3")
print("  - Multi-model ensemble with average scoring reduces single-model bias")
print("  - Majority voting provides a transparent, explainable final decision")


FINAL STATEMENT: Best Model and Pipeline

Most consistent model (lowest score variance) : cohere_rag_score
Most decisive model   (highest score variance): huggingface_rag_score

Final recommended candidate: 2 DV.txt

Best overall pipeline: RAG-enhanced (Week 4, chunk_size=300)
Reasoning:
  - Smaller chunks provide more precise semantic retrieval
  - Top-4 retrieval gives broader coverage than top-3
  - Multi-model ensemble with average scoring reduces single-model bias
  - Majority voting provides a transparent, explainable final decision


### Future Enhancements

Two key enhancements that would significantly improve this system beyond the current scope.

In [58]:
print("FUTURE ENHANCEMENT 1: Web-Based UI Upload System")
print("-" * 60)
print("""
Currently the system runs in Google Colab and requires manual file uploads.
A web-based UI would allow:
  - Recruiters to upload CVs and job descriptions via a browser
  - Real-time screening results displayed on screen
  - No coding knowledge required to use the system

Proposed stack: Streamlit or Gradio (Python-based, fast to build)
  - Streamlit: file_uploader widget for CVs and job description PDF
  - Results shown as interactive tables and score charts
  - One-click deployment to Streamlit Cloud (free hosting)

This directly addresses the Week 7-8 milestone of building a UI.
""")

print("FUTURE ENHANCEMENT 2: Bias and Fairness Checking")
print("-" * 60)
print("""
Current limitation: LLMs may introduce bias based on names, locations,
or universities mentioned in the resume.

Proposed solution:
  - Anonymise resumes before screening (remove names, gender indicators,
    nationality, age, photo references) using a pre-processing step
  - Compare scores before and after anonymisation to measure bias
  - Flag candidates where score changed significantly after anonymisation

This improves fairness and makes the system more suitable for real-world
HR deployment where anti-discrimination compliance is required.
""")


FUTURE ENHANCEMENT 1: Web-Based UI Upload System
------------------------------------------------------------

Currently the system runs in Google Colab and requires manual file uploads.
A web-based UI would allow:
  - Recruiters to upload CVs and job descriptions via a browser
  - Real-time screening results displayed on screen
  - No coding knowledge required to use the system

Proposed stack: Streamlit or Gradio (Python-based, fast to build)
  - Streamlit: file_uploader widget for CVs and job description PDF
  - Results shown as interactive tables and score charts
  - One-click deployment to Streamlit Cloud (free hosting)

This directly addresses the Week 7-8 milestone of building a UI.

FUTURE ENHANCEMENT 2: Bias and Fairness Checking
------------------------------------------------------------

Current limitation: LLMs may introduce bias based on names, locations,
or universities mentioned in the resume.

Proposed solution:
  - Anonymise resumes before screening (remove names, gen